# A visual walkthrough

This notebook illustrates the ideas in the [architecture overview](../explanation/architecture)
and the [mathematical foundations](../explanation/mathematical-foundations) on one small,
abstract problem: locating a source in 2D from noisy distance measurements to a few sensors.
The problem is deliberately simple so the *ideas* stay in view, but every step uses
photomancy's real engine, the same `build_scene_logdensity`, backends, `Posterior`,
`to_prior`, and expected information gain you would use on an orbit or an atmosphere.

With two sensors the problem is ambiguous: a source and its mirror image across the sensor
line fit the data equally well. That ambiguity is the thread that ties the walkthrough
together.


In [ ]:
import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

jax.config.update("jax_enable_x64", True)

from photomancy import LaplaceBackend, LaplaceMixtureBackend, build_scene_logdensity
from photomancy.eig import evaluate_candidates
from photomancy.priors import Uniform


class Source(eqx.Module):
    # the scene: a 2D source position (the only fitted leaf)
    pos: jnp.ndarray


sensors = jnp.array([[-3.0, 0.0], [3.0, 0.0]])  # two sensors on the x-axis
true_pos = jnp.array([1.0, 2.2])
sigma = 0.25
obs = jnp.linalg.norm(true_pos - sensors, axis=1) + sigma * jax.random.normal(
    jax.random.key(1), (sensors.shape[0],)
)


def forward(scene):
    return jnp.linalg.norm(scene.pos - sensors, axis=1)


def likelihood(pred):
    return -0.5 * jnp.sum(((pred - obs) / sigma) ** 2)


prior = Uniform(low=jnp.array([-5.0, -5.0]), high=jnp.array([5.0, 5.0]))
logdensity, z0, unravel = build_scene_logdensity(
    Source(pos=jnp.array([0.0, 0.0])), forward, likelihood, prior
)


def grid_density(ld, lim=5.0, n=160):
    xs = jnp.linspace(-lim, lim, n)
    X, Y = jnp.meshgrid(xs, xs)
    lp = jax.vmap(ld)(jnp.stack([X.ravel(), Y.ravel()], axis=1)).reshape(n, n)
    return np.array(X), np.array(Y), np.array(jnp.exp(lp - lp.max()))


def draw_ellipse(ax, mean, cov, **kw):
    vals, vecs = jnp.linalg.eigh(cov)
    ang = float(jnp.degrees(jnp.arctan2(vecs[1, 0], vecs[0, 0])))
    w, h = 2 * 2.0 * jnp.sqrt(jnp.maximum(vals, 0))
    ax.add_patch(Ellipse(np.array(mean), float(w), float(h), angle=ang, fill=False, **kw))


## 1. The fit

A fit is a logdensity over the fitted leaves of a scene. Here the scene is a `Source` with
one leaf, its 2D position, so `build_scene_logdensity` ravels that leaf into the flat
vector `z` the samplers see. The forward model returns the distances to the sensors, the
likelihood scores them against the data, and the prior is a broad box. Evaluating the
logdensity on a grid shows the posterior landscape: two equally good modes, the true source
and its mirror across the sensor line.


In [ ]:
X, Y, P = grid_density(logdensity)
fig, ax = plt.subplots(figsize=(5, 5))
ax.contourf(X, Y, P, levels=30, cmap="magma")
ax.scatter(*np.array(sensors).T, marker="^", c="cyan", s=80, label="sensors")
ax.scatter(*np.array(true_pos), marker="*", c="white", s=160, label="true source")
ax.set_title("posterior landscape (two sensors)")
ax.legend()
plt.show()


## 2. The Laplace approximation

The Laplace approximation finds a mode, then fits a Gaussian from the curvature there.
`LaplaceBackend` returns a `GaussianPosterior` carrying the mode, the covariance, and the
Laplace evidence. Starting near one mode captures it well, but a single Gaussian sees only
one mode, so it cannot represent the ambiguity on its own.


In [ ]:
post = LaplaceBackend(n_steps=200, min_eigenvalue=1e-6).run(logdensity, jnp.array([1.0, 2.0]))

fig, ax = plt.subplots(figsize=(5, 5))
ax.contourf(X, Y, P, levels=30, cmap="magma")
ax.scatter(*np.array(post.mean), c="cyan", s=60, label="MAP")
draw_ellipse(ax, post.mean, post.cov, edgecolor="cyan", lw=2)
ax.set_title(f"Laplace Gaussian (one mode), log Z = {float(post.evidence):.2f}")
ax.legend()
plt.show()


## 3. A mixture over the modes

Running the Laplace fit from several starts and weighting the modes by their evidence gives
a `MixturePosterior`. One start per basin recovers both modes here. Because the two sensors
lie on a line, the likelihood is exactly symmetric across it, so the two modes carry equal
evidence and equal weight.


In [ ]:
inits = jnp.array([[1.5, 2.0], [1.5, -2.0]])
mix = LaplaceMixtureBackend(n_steps=200, min_eigenvalue=1e-6).run(logdensity, inits)

print("modes:", np.round(np.array(mix.means), 2).tolist())
print("weights:", np.round(np.array(jnp.exp(mix.log_weights)), 3).tolist())
print("total log Z:", round(float(mix.evidence), 2))

fig, ax = plt.subplots(figsize=(5, 5))
ax.contourf(X, Y, P, levels=30, cmap="magma")
for k in range(mix.n_modes):
    draw_ellipse(ax, mix.means[k], mix.covs[k], edgecolor="cyan", lw=1.5)
ax.set_title("mixture: both modes")
plt.show()


## 4. Evidence and the Bayes factor

Treating the two modes as competing hypotheses, the ratio of their evidences is a Bayes
factor. With symmetric data it is one, so the log Bayes factor is zero: the data alone
cannot decide between the source and its mirror. This is the quantity an observation has to
move.


In [ ]:
log_bayes_factor = float(mix.log_evidences[0] - mix.log_evidences[1])
print("log Bayes factor (mode A vs mode B):", round(log_bayes_factor, 2))


## 5. Where to look next: expected information gain

The expected information gain scores a candidate observation by how much it is expected to
sharpen the belief. `evaluate_candidates` takes the mixture and a forward model for the
candidate, here the distance from the source to a candidate sensor position, and returns the
EIG over a grid of placements. A sensor on the baseline cannot tell the two mirror modes
apart, so the EIG is near zero there; the most informative placements sit off the baseline,
where the two modes predict very different distances.


In [ ]:
grid = np.stack(np.meshgrid(np.linspace(-4, 4, 25), np.linspace(-4, 4, 25)), -1).reshape(-1, 2)
candidates = jnp.array(grid)


def candidate_forward(z, c):
    return jnp.atleast_1d(jnp.linalg.norm(z - c))


eig = evaluate_candidates(mix, candidates, candidate_forward, sigma**2)
best = np.array(candidates[int(jnp.argmax(eig["total_eig"]))])
print("best next sensor:", np.round(best, 2).tolist())

fig, ax = plt.subplots(figsize=(5, 5))
ax.tricontourf(grid[:, 0], grid[:, 1], np.array(eig["total_eig"]), levels=30, cmap="viridis")
ax.scatter(*np.array(sensors).T, marker="^", c="white", s=80)
ax.scatter(*best, marker="X", c="red", s=140, label="best next sensor")
ax.set_title("expected information gain over candidate sensors")
ax.legend()
plt.show()


## 6. Observing and updating

Placing a sensor at the most informative spot and folding its measurement into the belief is
sequential Bayesian updating: the previous posterior becomes the next prior through
`to_prior`, and the new likelihood multiplies in. The single new measurement is consistent
with one mode and badly inconsistent with its mirror, so it carries a large Bayes factor and
the belief collapses onto the true source.


In [ ]:
new_sensor = jnp.array(best)
new_obs = jnp.linalg.norm(true_pos - new_sensor[None], axis=1) + sigma * jax.random.normal(
    jax.random.key(2), (1,)
)


def new_forward(scene):
    return jnp.linalg.norm(scene.pos - new_sensor[None], axis=1)


def new_likelihood(pred):
    return -0.5 * jnp.sum(((pred - new_obs) / sigma) ** 2)


# the previous posterior becomes the new prior
ld2, _, _ = build_scene_logdensity(
    Source(pos=jnp.array([0.0, 0.0])), new_forward, new_likelihood, mix.to_prior()
)
collapsed = LaplaceBackend(n_steps=300, min_eigenvalue=1e-6).run(ld2, jnp.array([0.0, 0.0]))

ll_a = float(new_likelihood(new_forward(Source(pos=mix.means[0]))))
ll_b = float(new_likelihood(new_forward(Source(pos=mix.means[1]))))
print("new-measurement log Bayes factor (mode A vs B):", round(ll_a - ll_b, 1))
print("collapsed posterior mean:", np.round(np.array(collapsed.mean), 2).tolist())

Xc, Yc, Pc = grid_density(ld2)
fig, ax = plt.subplots(figsize=(5, 5))
ax.contourf(Xc, Yc, Pc, levels=30, cmap="magma")
ax.scatter(*np.array(true_pos), marker="*", c="white", s=160, label="true source")
ax.scatter(*np.array(new_sensor), marker="^", c="red", s=80, label="new sensor")
ax.set_title("the belief collapses to one mode")
ax.legend()
plt.show()


## What this shows

Five objects carried the whole walkthrough: a logdensity over a scene, a backend that turns
it into a posterior, a mixture that holds the ambiguity, the evidence that scores competing
explanations, and the expected information gain that decides where to look next. None of it
was specific to source localization. Swapping the scene and the forward model for an orbit
or an atmosphere reuses every step unchanged, which is the point of the
[architecture](../explanation/architecture).
